# 2.4 Análise de Probabilidade com o Teorema de Bayes

In [1]:
import pandas as pd
df = pd.read_csv("US_Accidents_1_milhao.csv")

## Definição da variável alvo 

In [3]:
df["grave_alta"] = df["Severity"].apply(
    lambda valor: 1 if valor in [3, 4] else 0
)

In [4]:
df["Start_Time"] = pd.to_datetime(df["Start_Time"], errors="coerce")
df["hora"] = df["Start_Time"].dt.hour


def definir_periodo_dia(hora):
    if pd.isna(hora):
        return "desconhecido"
    elif 0 <= hora <= 5:
        return "madrugada"
    elif 6 <= hora <= 11:
        return "manha"
    elif 12 <= hora <= 17:
        return "tarde"
    else:
        return "noite"


df["periodo_dia"] = df["hora"].apply(definir_periodo_dia)

In [5]:
def definir_faixa_visibilidade(visibilidade):
    if pd.isna(visibilidade):
        return "desconhecida"
    elif visibilidade < 5:
        return "baixa"
    elif visibilidade < 10:
        return "media"
    else:
        return "alta"


df["faixa_visibilidade"] = df["Visibility(mi)"].apply(
    definir_faixa_visibilidade
)

In [6]:
def definir_precipitacao(precipitacao):
    if pd.isna(precipitacao) or precipitacao == 0:
        return "sem_precipitacao"
    else:
        return "com_precipitacao"


df["tem_precipitacao"] = df["Precipitation(in)"].apply(
    definir_precipitacao
)

In [7]:
df["clima"] = df["Weather_Condition"].fillna("desconhecido")
df["sunrise_sunset"] = df["Sunrise_Sunset"].fillna("desconhecido")


variaveis_preditoras = [
    "periodo_dia",
    "sunrise_sunset",
    "clima",
    "faixa_visibilidade",
    "tem_precipitacao",
    "Crossing",
    "Junction",
    "Traffic_Signal"
]

In [8]:
dados_bayes = df[variaveis_preditoras + ["grave_alta"]].dropna()


## Probabilidades a priori P(C)

In [10]:
probabilidades_priori = (
    dados_bayes["grave_alta"]
    .value_counts(normalize=True)
    .sort_index()
)

print("=" * 70)
print("PROBABILIDADES A PRIORI - P(C)")
print("=" * 70)

PROBABILIDADES A PRIORI - P(C)


In [11]:
for classe, probabilidade in probabilidades_priori.items():
    if classe == 0:
        nome_classe = "acidente leve"
    else:
        nome_classe = "acidente grave"

    print(f"P(C = {classe}) - {nome_classe}: {probabilidade:.4%}")

P(C = 0) - acidente leve: 64.4616%
P(C = 1) - acidente grave: 35.5384%


In [12]:
novo_dado = {
    "periodo_dia": "madrugada",
    "sunrise_sunset": "Night",
    "clima": "Rain",
    "faixa_visibilidade": "baixa",
    "tem_precipitacao": "com_precipitacao",
    "Crossing": True,
    "Junction": True,
    "Traffic_Signal": False
}


print("\n" + "=" * 70)
print("NOVO DADO INFORMADO PELO USUÁRIO - X")
print("=" * 70)


NOVO DADO INFORMADO PELO USUÁRIO - X


In [13]:
for variavel, valor in novo_dado.items():
    print(f"{variavel}: {valor}")


periodo_dia: madrugada
sunrise_sunset: Night
clima: Rain
faixa_visibilidade: baixa
tem_precipitacao: com_precipitacao
Crossing: True
Junction: True
Traffic_Signal: False


## 7. CÁLCULO DAS VEROSSIMILHANÇAS

In [28]:
resultados_nao_normalizados = {}

print("\n" + "=" * 70)
print("CÁLCULO DAS VEROSSIMILHANÇAS - P(X | C)")
print("=" * 70)


CÁLCULO DAS VEROSSIMILHANÇAS - P(X | C)


In [36]:
for classe in [0, 1]:
    dados_da_classe = dados_bayes[dados_bayes["grave_alta"] == classe]

    if classe == 0:
        nome_classe = "acidente leve"
    else:
        nome_classe = "acidente grave"

    print("\n" + "-" * 70)
    print(f"CLASSE {classe} - {nome_classe}")
    print("-" * 70)

    probabilidade_classe = probabilidades_priori[classe]

    print(
        f"Probabilidade a priori P(C = {classe}): "
        f"{probabilidade_classe:.6f}"
    )

    for variavel, valor_observado in novo_dado.items():
        total_registros_classe = len(dados_da_classe)

        quantidade_observada = len(
            dados_da_classe[
                dados_da_classe[variavel] == valor_observado
            ]
        )

        quantidade_valores_possiveis = dados_bayes[variavel].nunique()

        verossimilhanca = (
            quantidade_observada + 1
        ) / (
            total_registros_classe + quantidade_valores_possiveis
        )

        print(
            f"P({variavel} = {valor_observado} | C = {classe}) = "
            f"{verossimilhanca:.6f}"
        )

        probabilidade_classe *= verossimilhanca

    resultados_nao_normalizados[classe] = probabilidade_classe

    print(
        f"P(C = {classe}) * P(X | C) = "
        f"{probabilidade_classe:.12f}"
    )


----------------------------------------------------------------------
CLASSE 0 - acidente leve
----------------------------------------------------------------------
Probabilidade a priori P(C = 0): 0.644616
P(periodo_dia = madrugada | C = 0) = 0.088131
P(sunrise_sunset = Night | C = 0) = 0.267963
P(clima = Rain | C = 0) = 0.008124
P(faixa_visibilidade = baixa | C = 0) = 0.066884
P(tem_precipitacao = com_precipitacao | C = 0) = 0.055361
P(Crossing = True | C = 0) = 0.183357
P(Junction = True | C = 0) = 0.038095
P(Traffic_Signal = False | C = 0) = 0.709063
P(C = 0) * P(X | C) = 0.000000002268

----------------------------------------------------------------------
CLASSE 1 - acidente grave
----------------------------------------------------------------------
Probabilidade a priori P(C = 1): 0.355384
P(periodo_dia = madrugada | C = 1) = 0.082276
P(sunrise_sunset = Night | C = 1) = 0.277543
P(clima = Rain | C = 1) = 0.011460
P(faixa_visibilidade = baixa | C = 1) = 0.067093
P(tem_precipi

In [37]:
print("\nResultados não normalizados:")
print(resultados_nao_normalizados)


Resultados não normalizados:
{np.int64(1): np.float64(1.194429710652764e-09), 0: np.float64(2.268208371433062e-09)}


## 8. APLICAÇÃO DO TEOREMA DE BAYES

In [38]:
evidencia = sum(resultados_nao_normalizados.values())

probabilidades_posteriori = {}

for classe, valor in resultados_nao_normalizados.items():
    probabilidades_posteriori[classe] = valor / evidencia

## 9. PROBABILIDADES A POSTERIORI

In [39]:
print("\n" + "=" * 70)
print("PROBABILIDADES A POSTERIORI - P(C | X)")
print("=" * 70)

for classe, probabilidade in probabilidades_posteriori.items():
    if classe == 0:
        nome_classe = "acidente leve"
    else:
        nome_classe = "acidente grave"

    print(
        f"P(C = {classe} | X) - {nome_classe}: "
        f"{probabilidade:.4%}"
    )


PROBABILIDADES A POSTERIORI - P(C | X)
P(C = 1 | X) - acidente grave: 34.4948%
P(C = 0 | X) - acidente leve: 65.5052%


## 10. INTERPRETAÇÃO DO RESULTADO PROBABILÍSTICO

In [40]:
classe_prevista = max(
    probabilidades_posteriori,
    key=probabilidades_posteriori.get
)

probabilidade_prevista = probabilidades_posteriori[classe_prevista]

print("\n" + "=" * 70)
print("INTERPRETAÇÃO DO RESULTADO")
print("=" * 70)


INTERPRETAÇÃO DO RESULTADO


In [41]:
if classe_prevista == 0:
    print(
        "De acordo com o Teorema de Bayes, considerando as "
        "características informadas, o novo acidente possui maior "
        "probabilidade de pertencer à classe 0, ou seja, ser "
        "classificado como ACIDENTE LEVE."
    )
else:
    print(
        "De acordo com o Teorema de Bayes, considerando as "
        "características informadas, o novo acidente possui maior "
        "probabilidade de pertencer à classe 1, ou seja, ser "
        "classificado como ACIDENTE GRAVE."
    )

print(
    f"Probabilidade da classe prevista: "
    f"{probabilidade_prevista:.4%}"
)

De acordo com o Teorema de Bayes, considerando as características informadas, o novo acidente possui maior probabilidade de pertencer à classe 0, ou seja, ser classificado como ACIDENTE LEVE.
Probabilidade da classe prevista: 65.5052%


## TESTE COM UM CASO REAL DO DATASET

In [42]:
indice_caso_real = 10

caso_real = dados_bayes.iloc[indice_caso_real]

novo_dado_real = {
    "periodo_dia": caso_real["periodo_dia"],
    "sunrise_sunset": caso_real["sunrise_sunset"],
    "clima": caso_real["clima"],
    "faixa_visibilidade": caso_real["faixa_visibilidade"],
    "tem_precipitacao": caso_real["tem_precipitacao"],
    "Crossing": caso_real["Crossing"],
    "Junction": caso_real["Junction"],
    "Traffic_Signal": caso_real["Traffic_Signal"]
}

classe_real = caso_real["grave_alta"]

print("NOVO DADO REAL USADO NO TESTE")
print(novo_dado_real)

print("\nClasse real:")
if classe_real == 0:
    print("0 - acidente leve")
else:
    print("1 - acidente grave")

NOVO DADO REAL USADO NO TESTE
{'periodo_dia': 'manha', 'sunrise_sunset': 'Day', 'clima': 'Rain', 'faixa_visibilidade': 'media', 'tem_precipitacao': 'sem_precipitacao', 'Crossing': np.True_, 'Junction': np.True_, 'Traffic_Signal': np.False_}

Classe real:
1 - acidente grave


In [43]:
novo_dado = novo_dado_real

In [44]:
classe_prevista = max(
    probabilidades_posteriori,
    key=probabilidades_posteriori.get
)

print("\nCOMPARAÇÃO COM O CASO REAL")

print(f"Classe prevista pelo Bayes: {classe_prevista}")
print(f"Classe real no dataset: {classe_real}")

if classe_prevista == classe_real:
    print("Resultado: o Teorema de Bayes classificou corretamente este caso.")
else:
    print("Resultado: o Teorema de Bayes não classificou corretamente este caso.")


COMPARAÇÃO COM O CASO REAL
Classe prevista pelo Bayes: 0
Classe real no dataset: 1
Resultado: o Teorema de Bayes não classificou corretamente este caso.
